# 🗽 NYC Airbnb Room Type Classification & Multi-Model AI Suite

**Dataset:** [New York City Airbnb Open Data](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data) (Kaggle)

**Goal:** Predict the `room_type` of an Airbnb listing (`Entire home/apt`, `Private room`, `Shared room`) using tabular features, train an Artificial Neural Network (ANN), instantiate a Computer Vision CNN model, and export all production artifacts for deployment.

### 📋 Notebook Roadmap:
1. **Environment Setup & Imports**
2. **Data Ingestion** (KaggleHub / Direct Upload)
3. **Exploratory Data Analysis (EDA)**
4. **Data Cleaning & Outlier Capping**
5. **Stratified Train / Test Split**
6. **ColumnTransformer Preprocessing Pipeline**
7. **Candidate Machine Learning Algorithms Comparison**
8. **Final Production Pipeline Assembly**
9. **Test Set Evaluation & Metrics**
10. **Deep Learning ANN (Multi-Layer Perceptron) Training & Save (`ann_model.keras`)**
11. **Computer Vision CNN (MobileNetV2 Backbone) Build & Save (`cnn_room_vision.keras`)**
12. **One-Click Packaging & Auto-Download to PC (`nyc_airbnb_ai_models.zip`)**

## 📌 Cell 1: Import Core Libraries

In [ ]:
# Core Data Science & Plotting
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn Pipeline & Models
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Deep Learning & Computer Vision (TensorFlow / Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

# Colab Specific Auto-Download
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✅ Libraries loaded successfully! TensorFlow: {tf.__version__}, Scikit-Learn: ready.")

## 📌 Cell 2: Load the Dataset

In [ ]:
# Try downloading via KaggleHub, or fallback to local/uploaded file
csv_filename = "AB_NYC_2019.csv"

if not os.path.exists(csv_filename):
    try:
        import kagglehub
        path = kagglehub.dataset_download("dgomonov/new-york-city-airbnb-open-data")
        df = pd.read_csv(os.path.join(path, csv_filename))
        print("✅ Downloaded and loaded via KaggleHub:", path)
    except Exception as e:
        print("⚠️ KaggleHub failed or not authenticated, checking uploaded files...")
        if IN_COLAB and not os.path.exists(csv_filename):
            print("Please upload AB_NYC_2019.csv:")
            uploaded = files.upload()
        df = pd.read_csv(csv_filename)
else:
    df = pd.read_csv(csv_filename)
    print(f"✅ Loaded existing {csv_filename}")

print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 📌 Cell 3: Exploratory Data Analysis (EDA)

In [ ]:
print("--- Missing Values Check ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\n--- Target Distribution (room_type) ---")
print(df['room_type'].value_counts(normalize=True).round(4) * 100)

# Target Distribution Plot
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='room_type', palette='viridis')
plt.title("NYC Airbnb Room Type Distribution (Class Imbalance)")
plt.ylabel("Count")
plt.grid(axis='y', alpha=0.3)
plt.show()

## 📌 Cell 4: Data Cleaning & Percentile Outlier Capping

In [ ]:
# 1. Drop useless identifier columns that have no tabular generalizability
cols_to_drop = ['id', 'name', 'host_id', 'host_name', 'last_review']
df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns]).copy()

# 2. Missing reviews_per_month means 0 reviews yet, not missing data
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0.0)

# 3. Percentile Outlier Capping (99th percentile) instead of discarding data points
price_cap = df_clean['price'].quantile(0.99)
nights_cap = df_clean['minimum_nights'].quantile(0.99)

print(f"Capping 'price' at 99th percentile: ${price_cap:.2f}")
print(f"Capping 'minimum_nights' at 99th percentile: {nights_cap:.0f} nights")

df_clean['price'] = df_clean['price'].clip(upper=price_cap)
df_clean['minimum_nights'] = df_clean['minimum_nights'].clip(upper=nights_cap)

# 4. Separate Features (X) and Target (y)
X = df_clean.drop(columns=['room_type'])
y = df_clean['room_type']

print(f"Cleaned Feature Matrix X: {X.shape}, Target y: {y.shape}")

## 📌 Cell 5: Stratified Train / Test Split (80/20)

In [ ]:
# stratify=y preserves the exact class ratio (e.g. Shared room minority) in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"✅ Split Completed:")
print(f"Train Set: {X_train.shape[0]:,} samples")
print(f"Test Set:  {X_test.shape[0]:,} samples")

## 📌 Cell 6: ColumnTransformer Preprocessing Pipeline

In [ ]:
# Define Exact Columns matching Web App API & Production Interface
numerical_cols = [
    'latitude', 'longitude', 'price', 'minimum_nights',
    'number_of_reviews', 'reviews_per_month',
    'calculated_host_listings_count', 'availability_365'
]
categorical_cols = ['neighbourhood_group', 'neighbourhood']

# Numeric Pipeline: Median Imputation + Standard Scaling
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Most Frequent Imputation + OneHotEncoder
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle into a unified ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('numerical', num_pipeline, numerical_cols),
    ('categorical', cat_pipeline, categorical_cols)
])

print("✅ ColumnTransformer configured successfully for 8 numeric + 2 categorical features.")

## 📌 Cell 7: Candidate ML Algorithm Comparison

In [ ]:
# Compare Candidate Classifiers
candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=16, class_weight='balanced', random_state=42, n_jobs=-1)
}

print("Evaluating candidate algorithms on training subset (cross-validation score)...\n")
for name, clf in candidate_models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', clf)])
    # Quick 3-fold cross validation on sample to save time in Colab
    scores = cross_val_score(pipe, X_train.iloc[:10000], y_train.iloc[:10000], cv=3, scoring='accuracy')
    print(f"🔹 {name:<20}: Mean CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

## 📌 Cell 8: Final Production Pipeline Assembly & Training

In [ ]:
# Assemble the full final pipeline with RandomForestClassifier
final_classifier = RandomForestClassifier(
    n_estimators=200,
    min_samples_split=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', final_classifier)
])

print("Training Final Production Pipeline on 100% of Training Data...")
final_pipeline.fit(X_train, y_train)
print("✅ Final Pipeline training complete!")

## 📌 Cell 9: Test Set Evaluation & Metrics

In [ ]:
# Evaluate on the unseen Test Set
y_pred = final_pipeline.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"🎯 Final Test Set Accuracy: {test_acc * 100:.2f}%\n")

print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred, labels=final_pipeline.classes_)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=final_pipeline.classes_, 
            yticklabels=final_pipeline.classes_)
plt.title("Confusion Matrix (Test Set)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 📌 Cell 10: Deep Learning ANN (Artificial Neural Network) Training & Save (`ann_model.keras`)

In [ ]:
print("🚀 Training Deep Learning Multi-Layer Perceptron (ANN)...")

# Transform X_train & X_test into dense numerical matrices using preprocessor
X_train_trans = preprocessor.transform(X_train)
X_test_trans = preprocessor.transform(X_test)

if hasattr(X_train_trans, "toarray"):
    X_train_trans = X_train_trans.toarray()
    X_test_trans = X_test_trans.toarray()

# Encode target classes to integers & one-hot categorical
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

y_train_cat = to_categorical(y_train_num, num_classes=3)
y_test_cat = to_categorical(y_test_num, num_classes=3)

# Multi-Layer Perceptron ANN Architecture
ann_model = keras.Sequential([
    layers.Input(shape=(X_train_trans.shape[1],), name="tabular_input"),
    layers.Dense(128, activation='relu', name="dense_1"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', name="dense_2"),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu', name="dense_3"),
    layers.Dense(3, activation='softmax', name="softmax_output")
], name="NYC_Airbnb_ANN_Classifier")

ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train ANN for 15 epochs
ann_history = ann_model.fit(
    X_train_trans, y_train_cat,
    validation_data=(X_test_trans, y_test_cat),
    epochs=15,
    batch_size=128,
    verbose=1
)

# Save as standard Keras native model (.keras)
ann_model.save("ann_model.keras")
print("\n✅ Successfully saved: ann_model.keras")

## 📌 Cell 11: Computer Vision CNN Model Build & Save (`cnn_room_vision.keras`)

In [ ]:
print("🚀 Building Pretrained Computer Vision CNN Model (MobileNetV2 Backbone)...")

# MobileNetV2 for Room Image Quality & Scene Verification
base_vision = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_vision.trainable = False

cnn_model = keras.Sequential([
    layers.Input(shape=(224, 224, 3), name="room_photo_input"),
    base_vision,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax', name="scene_output") # bedroom, living_room, bathroom, kitchen, exterior
], name="NYC_Airbnb_CNN_Vision_Inspector")

cnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Save CNN model (.keras)
cnn_model.save("cnn_room_vision.keras")
print("✅ Successfully saved: cnn_room_vision.keras")

## 📌 Cell 12: Export All Artifacts & Auto-Download to Local PC (`nyc_airbnb_ai_models.zip`)

In [ ]:
# 1. Save Scikit-Learn Pipeline
joblib.dump(final_pipeline, "Model_Pipeline.pkl")
print("✅ Saved: Model_Pipeline.pkl")

# 2. Save Metadata
metadata = {
    "project_name": "NYC Airbnb Room Type Prediction & SaaS Intelligence",
    "version": "2.3.0",
    "classes": list(final_pipeline.classes_),
    "features": {
        "numerical": numerical_cols,
        "categorical": categorical_cols
    },
    "test_accuracy": float(round(test_acc, 4)),
    "ann_feature_dim": int(X_train_trans.shape[1]),
    "cnn_categories": ["bedroom", "living_room", "bathroom", "kitchen", "exterior"]
}

with open("model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print("✅ Saved: model_metadata.json")

# 3. Bundle everything into one zip file
zip_filename = "nyc_airbnb_ai_models.zip"
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write("Model_Pipeline.pkl")
    zipf.write("ann_model.keras")
    zipf.write("cnn_room_vision.keras")
    zipf.write("model_metadata.json")

file_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"\n🎉 All AI/ML models packaged successfully into '{zip_filename}' ({file_size_mb:.2f} MB)!")

# 4. Trigger Instant Browser Download to PC
if IN_COLAB:
    print("🚀 Triggering automatic browser download to your computer Downloads folder...")
    files.download(zip_filename)
else:
    print(f"File ready on disk at: {os.path.abspath(zip_filename)}")